In [ ]:
### Import packages using the scviEnv environment
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import warnings

warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

plt.rcParams['font.family'] = 'Arial'

In [ ]:
### Load the Meso data
adata_meso = sc.read_h5ad("../data/adata_all_samples_preprocessed.h5ad")
annotate = sc.read_h5ad('../data/adata_all_samples_annotated.h5ad')

### Then we add annotations
adata_meso.obs['celltype'] = annotate.obs.celltype.copy() 
adata_meso.obs['celltype_broad'] = annotate.obs.celltype_broad.copy()

### Where the celltype_broad equals neuronal_cells, we'll change it to schwann_cells
adata_meso.obs['celltype_broad'] = adata_meso.obs['celltype_broad'].replace(['Neuronal_Cells'], 'Schwann_Cells')

### Set the celltype_broad adipocytes to be split into white and brown, based on the celltype column
# Replace Adipocytes in celltype_broad based on celltype values
# Convert to string, make changes, convert back to category
adata_meso.obs['celltype_broad'] = adata_meso.obs['celltype_broad'].astype(str)
mask = adata_meso.obs['celltype_broad'] == 'Adipocytes'
adata_meso.obs.loc[mask, 'celltype_broad'] = adata_meso.obs.loc[mask, 'celltype']
adata_meso.obs['celltype_broad'] = adata_meso.obs['celltype_broad'].astype('category')

### Select only the Meso samples from the thoracic aorta PVAT
adata_meso = adata_meso[adata_meso.obs.tissue == 'Thoracic_Aorta'].copy()
### Add columns that exist in the taPVAT data but not in the Meso data, so that we can concatenate them later
adata_meso.obs['sex'] = 'M'

In [ ]:
### Load the taPVAT data
### You may download taPVAT data at GEO Accession ID GSE307140
filepath = "path/to/taPVAT_data/"  ### Note that the file path needs to be changed!
adata_tapvat = sc.read_h5ad(filepath + "taPVAT_8W_24W_M_F_HF_CTRL_combined_preprocessed.h5ad")
annotate = sc.read_h5ad(filepath + "taPVAT_combined_annotated_with_immune_fibro_ecs.h5ad")

### Then we add annotations and adjust to match the Meso data
adata_tapvat.obs['celltype'] = annotate.obs.celltype.copy() 
adata_tapvat.obs['celltype_broad'] = annotate.obs.celltype_broad.copy() 
adata_tapvat.obs['celltype'] = annotate.obs.celltype.copy() 

### Where the celltype equals Adipocytes_2 or adipocytes_3, we'll change it to adipocytes white
adata_tapvat.obs['celltype'] = adata_tapvat.obs['celltype'].replace(['Adipocytes_2', 'Adipocytes_3'], 'Adipocytes_White')
### Where the celltype equals pericytes or smcs, we'll change it to pericytes_smcs
adata_tapvat.obs['celltype_broad'] = adata_tapvat.obs['celltype_broad'].replace(['Pericytes', 'SMCs'], 'Pericytes_SMCs')
### Where the celltype_broad equals neuronal_cells, we'll change it to schwann_cells
adata_tapvat.obs['celltype_broad'] = adata_tapvat.obs['celltype_broad'].replace(['Neuronal_Cells'], 'Schwann_Cells')

### Set the celltype_broad adipocytes to be split into white and brown, based on the celltype column
# Replace Adipocytes in celltype_broad based on celltype values
# Convert to string, make changes, convert back to category
adata_tapvat.obs['celltype_broad'] = adata_tapvat.obs['celltype_broad'].astype(str)
mask = adata_tapvat.obs['celltype_broad'] == 'Adipocytes'
adata_tapvat.obs.loc[mask, 'celltype_broad'] = adata_tapvat.obs.loc[mask, 'celltype']
adata_tapvat.obs['celltype_broad'] = adata_tapvat.obs['celltype_broad'].astype('category')

### Drop the doublet celltype
adata_tapvat = adata_tapvat[adata_tapvat.obs['celltype'] != 'Doublets'].copy()

### Select only the 24W Male rats on control diet
adata_tapvat = adata_tapvat[adata_tapvat.obs.sample_type == 'taPVAT_Control_24W_M'].copy()
### Drop irrelevant columns

### Add columns that exist in the Meso data but not in the taPVAT data, so that we can concatenate them later
adata_tapvat.obs['platform'] = '10x'
adata_tapvat.obs['strain'] = 'Dahl_SS'
adata_tapvat.obs['species'] = 'Rattus_Norvegicus'
adata_tapvat.obs['tissue_type'] = 'Adipose'
adata_tapvat.obs['sample'] = adata_tapvat.obs['sample_id']
adata_tapvat.obs['sample_id'] = adata_tapvat.obs['Sample']


In [ ]:
### Combine the two datasets
adata = adata_meso.concatenate(adata_tapvat, batch_key = 'batch_key', join = 'outer', fill_value=0)
adata.var_names_make_unique()

In [ ]:
adata.obs.celltype.value_counts()

In [ ]:
adata.obs.celltype_broad.value_counts()

In [ ]:
df = adata.var.filter(like='gene_ids-', axis=1)
df

In [ ]:
### This fills in the missing values in gene_ids-0
df['combined'] = df['gene_ids-0'].fillna(df['gene_ids-1'])
### If the above function worked properly, this will return False.
df.combined.isna().unique()

In [ ]:
#There's no need to have so many columns in adata.var, so let's drop them
adata.var = adata.var.drop(adata.var.columns, axis = 1)

In [ ]:
#Now we'll add back the one's we need
adata.var['gene_ids'] = df['combined']

In [ ]:
#recalculate QC metrics
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("Mt-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))
# hemoglobin genes.
adata.var["hb"] = adata.var_names.str.contains(("^Hb[^(P)]"))

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], 
                           inplace=True, percent_top=[20], log1p=True)

In [ ]:
adata.var

In [ ]:
adata

In [ ]:
p1 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="tissue")
p2 = sc.pl.violin(adata, "pct_counts_mt")

In [ ]:
print(f"Total number of genes: {adata.n_vars}")
# Min 1 cells - filters out 0 count genes
sc.pp.filter_genes(adata, min_cells=3)
print(f"Number of genes after cell filter: {adata.n_vars}")

## Filter cells
print(f"Total number of cells: {adata.n_obs}")
sc.pp.filter_cells(adata, min_genes = 100)
print(f"Number of cells after gene filter: {adata.n_obs}")

In [ ]:
sc.pl.scatter(adata, x='log1p_total_counts', y='pct_counts_mt', color = 'tissue')
sc.pl.scatter(adata, x='log1p_total_counts', y='pct_counts_ribo', color = 'tissue')
sc.pl.scatter(adata, x='log1p_total_counts', y='log1p_n_genes_by_counts', color = 'tissue')
sc.pl.scatter(adata, x='log1p_total_counts', y='scDblFinder_score', color = 'tissue')

In [ ]:
sc.pl.violin(adata, keys=['log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_mt', 'pct_counts_ribo'], groupby='tissue', rotation=90, stripplot = False, inner = 'box')

In [ ]:
result = pd.DataFrame(adata.obs[['sample', 'sample_type']].value_counts().reset_index())
result.columns = ['sample', 'sample_type', 'total_count']
result

In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.barplot(x='sample_type', y='total_count', data=result, capsize = 0.1)
plt.title('Number of Cells per Sample Type Pre-QC')
plt.ylabel('Count')
plt.xlabel('')
plt.xticks(rotation=15)

plt.show()

In [ ]:
### We'll normalize and log transform the data in adata here, and then we'll use the raw attribute to store the normalized and log-transformed data for later use in LIANA.
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

In [ ]:
adata.write_h5ad('../data/adata_meso_tapvat_combined_preprocessed.h5ad')